In [ ]:
#| default_exp app_pages

In [ ]:
#| export

from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
from fastlite import *
import fasthtml.components as fc
from fasthtml.common import A, Button as FhButton, I, Span
from fh_matui.foundations import normalize_tokens, stringify, VEnum
from fh_matui.core import *
from fh_matui.components import *



In [ ]:
#| code-fold: true
#| eval: false

from fasthtml.jupyter import *
from IPython.display import HTML, Markdown, Image
import socket
import time
import subprocess

def kill_process_on_port(port):
    """Kill any process using the specified port on Windows"""
    try:
        # Find process using the port
        result = subprocess.run(
            f'netstat -ano | findstr :{port}',
            shell=True, capture_output=True, text=True
        )
        
        if result.stdout:
            # Extract PID from netstat output
            lines = result.stdout.strip().split('\n')
            for line in lines:
                if 'LISTENING' in line:
                    pid = line.strip().split()[-1]
                    subprocess.run(f'taskkill /PID {pid} /F', shell=True, capture_output=True)
                    print(f"✓ Killed process {pid} on port {port}")
                    time.sleep(0.5)
                    return True
        return False
    except Exception as e:
        print(f"⚠ Could not kill process on port {port}: {e}")
        return False

def find_available_port(start_port=3333, max_attempts=10):
    """Find an available port starting from start_port"""
    for port in range(start_port, start_port + max_attempts):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            try:
                s.bind(('', port))
                return port
            except OSError:
                continue
    raise RuntimeError(f"Could not find an available port in range {start_port}-{start_port+max_attempts}")

# Stop existing server if running
if 'server' in globals(): 
    try:
        server.stop()
        time.sleep(0.5)
    except:
        pass

# Try to kill any process on preferred port, then find available port
preferred_port = 9999
kill_process_on_port(preferred_port)
port = find_available_port(preferred_port)

app = FastHTML(hdrs=(Theme.blue.headers(title="fastmaterial", mode="dark")))
rt = app.route

try:
    server = JupyUvi(app, port=port)
    preview = partial(HTMX, app=app, port=port)
    print(f"✓ Server running on port {port}")
except Exception as e:
    print(f"✗ Failed to start server: {e}")
    raise

✓ Server running on port 9999


## Login Page

In [ ]:
#| export
def LoginScreen(
    title='Sign In',
    subtitle='Choose your preferred sign-in method',
    providers=None,      
    left_slot=None,      
    logo_src=None,       
    testimonial_text=None, 
    brand_bg_cls='primary',
    left_cols=9,  # Number of columns for left side (out of 12), default 9 = 75%
    cls='',
    **kwargs
):
    """
    A configurable Split Login Screen using strict 'Fat Matt UI' helpers.
    
    Args:
        left_cols: Number of columns (out of 12) for the left brand section. 
                   Common values: 6 (50%), 7 (58%), 8 (67%), 9 (75%), 10 (83%)
    """
    
    # 1. Defaults
    if providers is None:
        providers = [
            {'label': 'Continue with Google', 'icon': 'https://authjs.dev/img/providers/google.svg', 'href': '/auth/google', 'cls': 'border responsive surface'},
            {'label': 'Continue with GitHub', 'icon': 'https://authjs.dev/img/providers/github.svg', 'icon_cls': 'invert', 'href': '/auth/github', 'cls': 'fill responsive inverse-surface'}
        ]

    # 2. Build Left Column
    if left_slot:
        left_content = left_slot
    else:
        default_content = []
        if logo_src: 
            default_content.append(Img(src=logo_src, cls="responsive margin-bottom", style="max-height: 60px"))
        if not logo_src and not testimonial_text:
            default_content.append(H3("Welcome", cls="center-align white-text"))
        if testimonial_text:
            default_content.append(Blockquote(P(f'"{testimonial_text}"', cls="italic center-align white-text")))
            
        # USER HELPER: DivCentered handles the alignment and spacing
        # Wrap in a Div with max-width, then center it
        left_content = Div(
            DivCentered(*default_content),
            style="max-width: 400px"
        )

    # 3. Build Right Column Buttons
    button_list = []
    for p in providers:
        icon = Img(src=p['icon'], cls=f"circle tiny spacing-right {p.get('icon_cls', '')}") if p.get('icon') else ""
        button_list.append(
            A(
                Button(icon, Span(p['label']), cls=p.get('cls')), 
                href=p.get('href', '#'),
                style="text-decoration: none; display: block;" 
            )
        )

    # USER HELPER: DivVStacked handles the vertical list + gap
    auth_buttons = DivVStacked(
        *button_list,
        cls="w-full"
    )

    # Wrap in a Div with max-width, then center it
    right_content = Div(
        DivCentered(
            H4(title, cls="center-align bold margin-bottom"),
            P(subtitle, cls="center-align medium-text margin-bottom"),
            auth_buttons,
        ),
        style="max-width: 350px; width: 100%"
    )

    # 4. The Grid - Calculate column sizes based on percentage
    left_percent = (left_cols / 12) * 100
    right_percent = ((12 - left_cols) / 12) * 100
    
    # Use CSS Grid for precise percentage-based split (BeerCSS compatible)
    return Div(
        # Left (Brand) - Configurable width with primary background
        Div(
            DivCentered(left_content),
            cls=f"{brand_bg_cls} padding",
            style="display: flex; align-items: center; justify-content: center; min-height: 100vh;"
        ),
        # Right (Auth) - Remaining width
        Div(
            DivCentered(right_content),
            cls="padding",
            style="display: flex; align-items: center; justify-content: center; min-height: 100vh;"
        ),
        # Use CSS Grid to control the split with configurable percentages
        cls=cls,
        style=f"display: grid; grid-template-columns: {left_percent}% {right_percent}%; min-height: 100vh; margin: 0; padding: 0;",
        **kwargs
    )

In [ ]:
#| code-fold: true
#| eval: false


preview(LoginScreen())

In [ ]:
#| code-fold: true
#| eval: false

@app.get("/test-login")
def login():
    return LoginScreen()

## Index Page

In [ ]:
#| code-fold: true
#| eval: false

def nav_items():
    """Returns navigation items for NavBar"""
    return [
        A("Home", href='/', hx_get='/', hx_target='#main-content', hx_push_url='true'),
        A("Dashboard", href='/dashboard', hx_get='/dashboard', hx_target='#main-content', hx_push_url='true'),
        A("Cookies", href='/cookies', hx_get='/cookies', hx_target='#main-content', hx_push_url='true'),
        A(Icon('light_mode'), cls='circle', onclick='toggleMode()', title='Toggle dark/light mode', style='margin-left: 1rem;'),
    ]


def sidebar_items():
    """Returns navigation items for sidebar"""
    return [
        A(Icon('home'), Span('Home'), href='/', hx_get='/', hx_target='#main-content', hx_push_url='true'),
        A(Icon('dashboard'), Span('Dashboard'), href='/dashboard', hx_get='/dashboard', hx_target='#main-content', hx_push_url='true'),
        A(Icon('cookie'), Span('Cookies'), href='/cookies', hx_get='/cookies', hx_target='#main-content', hx_push_url='true'),
        A(Icon('logout'), Span('Logout'), href='/login', hx_get='/login', hx_target='#main-content', hx_push_url='true'),
    ]

# Login page - starting point
@rt('/')
@rt('/login')
def get(req):
    # Update providers to route to dashboard after "login"
    return LoginScreen(
        providers=[
            {'label': 'Continue with Google', 'icon': 'https://authjs.dev/img/providers/google.svg', 'href': '/dashboard', 'cls': 'border responsive surface'},
            {'label': 'Continue with GitHub', 'icon': 'https://authjs.dev/img/providers/github.svg', 'icon_cls': 'invert', 'href': '/dashboard', 'cls': 'fill responsive inverse-surface'}
        ]
    )

# Dashboard - main admin panel after login
@rt('/dashboard')
def get(req):
    content = Div(
        H1("Dashboard"),
        P("Welcome to the admin panel!", cls='grey-text'),
        Grid(
            Card(H6("Total Users"), H2("1,234"), cls='padding'),
            Card(H6("Active Sessions"), H2("456"), cls='padding'),
            Card(H6("Page Views"), H2("12.5K"), cls='padding'),
            Card(H6("Conversion Rate"), H2("3.2%"), cls='padding'),
            cols=4
        ),
        Div(style='margin-top: 2rem;')(
            Card(
                H5("Recent Activity"),
                Ul(
                    Li("User john@example.com logged in"),
                    Li("New order #1234 created"),
                    Li("Payment processed for order #1233"),
                    Li("User jane@example.com signed up")
                ),
                cls='padding'
            )
        ),
        cls='padding'
    )
    
    if 'HX-Request' in req.headers:
        return content
    
    return Layout(
        Div(content, id='main-content'),
        sidebar_links=sidebar_items(),
        nav_bar=NavBar(*nav_items(), brand=H3('Admin Panel'), sticky=True)
    )

# Cookies banner demo
@rt('/cookies')
def get(req):
    content = Div(
        H2("Cookie Banner Examples"),
        P("Scroll to the bottom to see the cookie banner"),
        Div(style='height: 70vh;'),
        CookiesBanner(),
        cls='padding'
    )
    
    if 'HX-Request' in req.headers:
        return content
    
    return Layout(
        Div(content, id='main-content'),
        sidebar_links=sidebar_items(),
        nav_bar=NavBar(*nav_items(), brand=H3('Admin Panel'), sticky=True)
    )

In [ ]:
#| hide

import nbdev as nb
nb.nbdev_export()